In [ ]:
%load_ext autoreload
%autoreload 2

%load_ext rich

In [ ]:
from __future__ import annotations

import json
import mimetypes
import os
import re
import time
from pathlib import Path
from typing import ClassVar, Iterable, Literal

import ollama
import pandas as pd
import requests
import yaml
from ollama import ChatResponse
from pydantic import BaseModel, ValidationError, model_validator
from tqdm import tqdm

from aymurai.utils.json_data import load_json, save_json
from aymurai.utils.yaml_data import load_yaml

## Configuración

In [ ]:
MODEL = "gemma4"
DEVICE = "cuda"

API_BASE_URL = os.getenv("API_BASE_URL", "https://aymurai.collectiveai.io/api")
ENDPOINT = f"{API_BASE_URL}/misc/document-extract"
DATA_ROOT = Path(
    os.getenv("DOCUMENT_DATA_ROOT", "../../../resources/data/restricted/defensoria/pdfs")
)
DOC_EXTENSIONS = {".pdf", ".docx"}
REQUEST_TIMEOUT = float(os.getenv("REQUEST_TIMEOUT", "30"))

PROMPT_CONFIG_PATH = Path("../../../resources/llm/defensoria_extractor.yml")
RESULTS_PATH = Path(f"./information-extraction-results-{DEVICE}.json")

print(f"Model:           {MODEL}")
print(f"Prompt config:   {PROMPT_CONFIG_PATH}")
print(f"Results output:  {RESULTS_PATH}")
print(f"Target endpoint: {ENDPOINT}")

## System prompt desde YAML

In [ ]:
config = load_yaml(PROMPT_CONFIG_PATH)
config.keys()

In [ ]:
def build_fields_block(fields: dict) -> str:
    """Convert the fields dict from the YAML config into a Markdown block for the prompt.

    Args:
        fields (dict): field_name -> description mapping, as it comes from the YAML.

    Returns:
        str: Markdown list ("- **field**: description"), one line per field.
    """
    lines = []
    for field, description in fields.items():
        desc = str(description).strip().replace("\n", " ")
        lines.append(f"- **{field}**: {desc}")
    return "\n".join(lines)


def build_system_prompt(config: dict) -> str:
    """Build the full system prompt from the loaded YAML config.

    Args:
        config (dict): Parsed defensoria_extractor.yml content (with the
            "system-prompts", "fields", "taxonomy", "output-format" keys).

    Returns:
        str: Final system prompt, with the taxonomy and output schema embedded.
    """
    prompts = config["system-prompts"]
    fields = config["fields"]
    taxonomy = config["taxonomy"]
    schema = config["output-format"]["schema"]

    schema_json = json.dumps(schema, ensure_ascii=False, indent=2)

    return f"""{prompts["information-extraction"]}

{prompts["extraction-guidelines"]}
# Campos a extraer

{build_fields_block(fields)}

# Taxonomía de temas y subtemas

{taxonomy}

# Formato de salida (JSON)

Responde exclusivamente con un objeto JSON válido con la siguiente estructura:

```json
{schema_json}
```
"""


system_prompt = build_system_prompt(config)
print(system_prompt)

## Modelos Pydantic

In [ ]:
# Produces: {tema: frozenset(subtemas)}
_taxonomy: dict[str, frozenset[str]] = {
    tema: frozenset(subtemas)
    for entry in yaml.safe_load(config["taxonomy"])
    for tema, subtemas in entry.items()
}

Tema = Literal[tuple(_taxonomy)]


class Destinatario(BaseModel):
    nombre: str | None = None
    cargo: str | None = None
    destinatario_principal: bool
    sector: str | None = None

class DataExtraction(BaseModel):
    numero_recomendacion: str | None = None
    fecha_recomendacion: str | None = None
    destinatarios: list[Destinatario]
    tema: Tema | None = None
    subtema: str | None = None
    # destinatarios_por_sector: list[str]
    datos_personales: bool
    contenido_para_publicar: str

    taxonomy: ClassVar[dict[str, frozenset[str]]] = _taxonomy

    @model_validator(mode="after")
    def validate_tema_subtema(self) -> DataExtraction:
        if self.subtema is None:
            return self

        if self.tema is None:
            raise ValueError("No se puede especificar un subtema sin un tema")

        valid_subtemas = self.taxonomy[self.tema]

        if self.subtema not in valid_subtemas:
            raise ValueError(
                f"El subtema {self.subtema!r} no pertenece al tema "
                f"{self.tema!r}. Subtemas válidos: {sorted(valid_subtemas)}"
            )

        return self


DataExtraction.model_json_schema()

## Carga de documentos

In [ ]:
if not DATA_ROOT.exists():
    raise FileNotFoundError(
        f"Directory '{DATA_ROOT}' not found. Update DATA_ROOT before continuing."
    )


def discover_documents(root: Path, extensions: Iterable[str]) -> list[Path]:
    """Recursively find the documents to process under `root`.

    Args:
        root (Path): Root folder to search (recursively).
        extensions (Iterable[str]): Extensions to include (with or without leading dot).

    Returns:
        list[Path]: Matching paths, sorted alphabetically.
    """
    extensions = {ext.lower() for ext in extensions}
    return sorted(
        path
        for path in root.rglob("*")
        if path.is_file() and path.suffix.lower() in extensions
    )


documents = discover_documents(DATA_ROOT, DOC_EXTENSIONS)
print(f"Discovered {len(documents)} documents.")

In [ ]:
def call_extraction_api(
    session: requests.Session, file_path: Path
) -> dict[str, object]:
    """Upload a document to ENDPOINT (/misc/document-extract) and return its text.

    Args:
        session (requests.Session): HTTP session reused across calls.
        file_path (Path): Local path of the file to upload.

    Returns:
        dict[str, object]: {"path", "status" ("success"/"failure"), "status_code",
        "elapsed_s", "detail"} -- if status="success", detail holds
        {"document_id", "document"}; otherwise it holds the error detail.
    """
    payload: dict[str, object] = {
        "path": str(file_path),
        "status": "failure",
        "status_code": None,
        "elapsed_s": None,
        "detail": None,
    }

    if not file_path.exists():
        payload["detail"] = "File does not exist"
        return payload

    mime_type = mimetypes.guess_type(file_path.name)[0] or "application/octet-stream"
    files = {"file": (file_path.name, file_path.open("rb"), mime_type)}

    try:
        start = time.perf_counter()
        response = session.post(ENDPOINT, files=files, timeout=REQUEST_TIMEOUT)
        elapsed = time.perf_counter() - start
    except requests.RequestException as exc:
        payload["detail"] = f"Request failed: {exc}"
        return payload
    finally:
        files["file"][1].close()

    payload["status_code"] = response.status_code
    payload["elapsed_s"] = elapsed

    try:
        response_body = response.json()
    except ValueError:
        response_body = {"raw": response.text[:500]}

    if response.ok:
        payload["status"] = "success"
        payload["detail"] = {
            "document_id": response_body.get("document_id"),
            "document": response_body.get("document", []),
        }
    else:
        payload["detail"] = response_body

    return payload

In [ ]:
# Test rápido con un solo documento (2 conserva el cargo)
doc_path = documents[25]
extracted_document = call_extraction_api(requests.Session(), doc_path)["detail"]
document_text = "\n".join(extracted_document["document"])
print(f"Documento: {doc_path.name}")
print(f"Caracteres: {len(document_text)}")
print(document_text[:500])

## Inferencia

In [ ]:
def parse_json_object(raw_output: str) -> dict:
    """Extract the first valid JSON object from the model's response.

    Args:
        raw_output (str): Raw text returned by the LLM (may include markdown
            fences or extra text around the JSON).

    Raises:
        ValueError: If no valid JSON object is found.

    Returns:
        dict: The first JSON object parsed from the text.
    """
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", raw_output.strip())
    decoder = json.JSONDecoder()

    for start, char in enumerate(cleaned):
        if char != "{":
            continue
        try:
            parsed, _ = decoder.raw_decode(cleaned[start:])
        except json.JSONDecodeError:
            continue
        if isinstance(parsed, dict):
            return parsed

    raise ValueError("No valid JSON object found in model response")


def get_chat_response(
    system_prompt: str,
    user_prompt: str,
    model: str = MODEL,
    options: dict = {"num_ctx": 32_768, "num_predict": 8192},
) -> ChatResponse:
    """Call ollama.chat with a system prompt and a user message.

    Args:
        system_prompt (str): System prompt to use.
        user_prompt (str): User message (document text).
        model (str): Ollama model name. Defaults to MODEL.
        options (dict): Ollama generation options (num_ctx, num_predict, etc.).

    Returns:
        ChatResponse: Raw response from ollama.chat.
    """
    return ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        options=options,
    )


def build_validation_retry_prompt(
    system_prompt: str,
    validation_error: str,
    previous_output: str,
) -> str:
    """Build a retry system prompt asking the model to fix the validation error.

    Args:
        system_prompt (str): Original system prompt.
        validation_error (str): Error message from the failed validation attempt.
        previous_output (str): The model's previous (invalid) response.

    Returns:
        str: System prompt extended with the correction instructions.
    """
    return f"""{system_prompt}

# Corrección obligatoria

Tu respuesta anterior no pasó la validación automática.
Devuelve exclusivamente un objeto JSON válido, sin Markdown ni texto adicional.

Error de validación:
{validation_error}

Respuesta anterior:
{previous_output}
"""


def evaluate_chat_response(
    system_prompt: str,
    user_prompt: str,
    model: str = MODEL,
    options: dict = {"num_ctx": 32_768, "num_predict": 8192},
    validation_model: type[BaseModel] | None = None,
    max_retries: int = 2,
) -> dict:
    """Call the model, record metrics, and validate against a Pydantic model.

    Retries up to max_retries times, feeding the validation error back to the model.

    Args:
        system_prompt (str): Initial system prompt.
        user_prompt (str): User message (document text).
        model (str): Ollama model name. Defaults to MODEL.
        options (dict): Ollama generation options.
        validation_model (type[BaseModel] | None): Pydantic model to validate
            the parsed response against. If None, doesn't validate or retry.
        max_retries (int): Max retries in addition to the first attempt.

    Returns:
        dict: Metrics and result of the last attempt -- includes "chat_response",
        token counts/durations, "attempts" (all attempts), "validation_succeeded",
        "validation_error", "parsed_response", and "validated_response".
    """
    attempts = []
    current_system_prompt = system_prompt

    for attempt_idx in range(max_retries + 1):
        start = time.time()
        response = get_chat_response(
            system_prompt=current_system_prompt,
            user_prompt=user_prompt,
            model=model,
            options=options,
        )
        end = time.time()

        input_tokens = getattr(response, "prompt_eval_count", None)
        output_tokens = getattr(response, "eval_count", None)
        total_tokens = (
            input_tokens + output_tokens
            if input_tokens is not None and output_tokens is not None
            else None
        )
        input_duration_ms = (
            response.prompt_eval_duration / 1_000_000
            if hasattr(response, "prompt_eval_duration")
            else None
        )
        output_duration_ms = (
            response.eval_duration / 1_000_000
            if hasattr(response, "eval_duration")
            else None
        )
        model_total_duration_ms = (
            response.total_duration / 1_000_000
            if hasattr(response, "total_duration")
            else None
        )
        tokens_per_second = (
            output_tokens / (output_duration_ms / 1000)
            if output_tokens and output_duration_ms
            else None
        )

        attempt_result = {
            "attempt": attempt_idx + 1,
            "chat_response": response.message.content,
            "input_tokens": input_tokens,
            "input_duration_ms": input_duration_ms,
            "output_tokens": output_tokens,
            "output_duration_ms": output_duration_ms,
            "total_tokens": total_tokens,
            "model_duration_ms": model_total_duration_ms,
            "measured_duration_ms": (end - start) * 1000,
            "tokens_per_second": tokens_per_second,
            "raw_response": response.model_dump(),
        }

        if validation_model is None:
            attempts.append(attempt_result)
            break

        try:
            parsed = parse_json_object(response.message.content)
            validated = validation_model.model_validate(parsed)
        except (json.JSONDecodeError, ValidationError, ValueError) as exc:
            last_error = str(exc)
            attempt_result.update(
                {"validation_succeeded": False, "validation_error": last_error}
            )
            attempts.append(attempt_result)
            if attempt_idx >= max_retries:
                break
            current_system_prompt = build_validation_retry_prompt(
                system_prompt=system_prompt,
                validation_error=last_error,
                previous_output=response.message.content,
            )
            continue

        attempt_result.update(
            {
                "validation_succeeded": True,
                "validation_error": None,
                "parsed_response": parsed,
                "validated_response": validated.model_dump(),
            }
        )
        attempts.append(attempt_result)
        break

    final = attempts[-1]
    return {
        "model": model,
        "options": options,
        "system_prompt": system_prompt,
        "user_prompt": user_prompt,
        **{
            k: final[k]
            for k in [
                "chat_response",
                "input_tokens",
                "input_duration_ms",
                "output_tokens",
                "output_duration_ms",
                "total_tokens",
                "model_duration_ms",
                "measured_duration_ms",
                "tokens_per_second",
                "raw_response",
            ]
        },
        "attempts": attempts,
        "num_attempts": len(attempts),
        "validation_succeeded": final.get("validation_succeeded"),
        "validation_error": final.get("validation_error"),
        "parsed_response": final.get("parsed_response"),
        "validated_response": final.get("validated_response"),
    }

In [ ]:
import unicodedata

from rapidfuzz import fuzz, process

ORGANIGRAM_PATH = Path("organigram_GCBA.json")

with ORGANIGRAM_PATH.open(encoding="utf-8") as f:
    organigram = json.load(f)


def normalize_text(value) -> str:
    """Lowercase + strip accents, so fuzzy matching ignores casing/tildes.

    Args:
        value: Text to normalize (or None/NaN).

    Returns:
        str: Normalized text, or "" if value is None/NaN.
    """
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    value = str(value).strip().lower()
    value = unicodedata.normalize("NFKD", value)
    return "".join(char for char in value if not unicodedata.combining(char))


CARGO_ROLE_PREFIXES = (
    "ministro", "ministra",
    "secretario", "secretaria",
    "subsecretario", "subsecretaria",
    "director general", "directora general",
    "director", "directora",
    "presidente", "presidenta",
)


CARGO_ROLE_PATTERN = re.compile(r"^(" + "|".join(CARGO_ROLE_PREFIXES) + r")\s+(?:de\s+)?")


# del/de la/de only counts as the parent-chain boundary when followed by an
# actual parent-institution noun -- otherwise "del"/"de la" inside a compound
# office name (e.g. "Registro del Estado Civil") gets mistaken for it.
PARENT_INSTITUTION_KEYWORDS = (
    "ministerio", "secretaria", "subsecretaria", "gobierno",
    "jefatura", "agencia", "ente", "sindicatura", "procuracion",
)
CARGO_PARENT_MARKERS = re.compile(
    r"\s*,?\s+(?:dependiente\s+)?(?:del|de la|de)\s+"
    r"(?=(?:" + "|".join(PARENT_INSTITUTION_KEYWORDS) + r")\b)"
)


CARGO_NOISE_PATTERNS = [
    r"\bgobierno de la ciudad autonoma de buenos aires\b",
    r"\bgobierno de la ciudad de buenos aires\b",
    r"\bciudad autonoma de buenos aires\b",
    r"\bciudad de buenos aires\b",
]


def extract_cargo_role(text: str) -> str:
    """Keep only the "<role> de <topic>" clause, dropping the "del/dependiente
    de/de la ..." parent-institution chain that follows a role title.

    Args:
        text (str): Already-normalized cargo text (normalize_text applied).

    Returns:
        str: Text with the parent-institution chain removed, if a role
        prefix was found; otherwise the text unchanged.
    """
    match = CARGO_ROLE_PATTERN.match(text)
    if not match:
        return text
    rest = text[match.end():]
    parent_match = CARGO_PARENT_MARKERS.search(rest)
    topic = (rest[: parent_match.start()] if parent_match else rest).strip()
    # topic can itself start with "del"/"de"/"dependiente" (e.g. "del Registro
    # del Estado Civil"); only prepend "de" when it isn't already there, or
    # reconstruction would produce a duplicated "de del ...".
    first_word = topic.split(" ", 1)[0] if topic else ""
    if first_word not in ("de", "del", "dependiente"):
        topic = f"de {topic}"
    return f"{match.group(1)} {topic}".strip()


def _strip_noise_and_trailing(text: str) -> str:
    """Remove GCBA boilerplate phrases and any dangling trailing preposition.

    Args:
        text (str): Already-normalized cargo text.

    Returns:
        str: Cleaned text.
    """
    for pattern in CARGO_NOISE_PATTERNS:
        text = re.sub(pattern, " ", text)
    text = re.sub(r"\b(del|de la|de|dependiente(?: del| de la| de)?)\s*$", "", text.strip())
    return re.sub(r"\s+", " ", text).strip()


def strip_cargo_boilerplate(value) -> str:
    """normalize_text plus GCBA boilerplate removal only (no role extraction).

    Args:
        value: Raw cargo text.

    Returns:
        str: Normalized text with GCBA boilerplate removed (keeps the
        parent-institution chain).
    """
    return _strip_noise_and_trailing(normalize_text(value))


def clean_cargo(value) -> str:
    """normalize_text, plus role-prefix extraction and GCBA boilerplate removal.

    Args:
        value: Raw cargo text.

    Returns:
        str: Cleaned text, ready for matching (role + topic, without the
        parent-institution chain or GCBA boilerplate).
    """
    return _strip_noise_and_trailing(extract_cargo_role(normalize_text(value)))


def iter_people_with_dependency(node, parent=None, path=None):
    """Recursively walk the organigram JSON, yielding one person at a time.

    Args:
        node: Current organigram node (dict) or list of nodes.
        parent: Immediate parent node ({"cargo", "sigla"}), if any.
        path: Cargo names from the root down to `node`, used to build ruta_cargos.

    Yields:
        dict: {"nombre", "cargo", "sigla", "depende_de_cargo",
        "depende_de_sigla", "ruta_cargos"} for each person under `node`.
    """
    if path is None:
        path = []

    if isinstance(node, list):
        for child in node:
            yield from iter_people_with_dependency(child, parent=parent, path=path)
        return

    if not isinstance(node, dict):
        return

    cargo = str(node.get("cargo", "")).strip()
    sigla = str(node.get("sigla", "")).strip()
    nombre = str(node.get("nombre", "")).strip()
    current_path = [*path, cargo] if cargo else path

    if nombre:
        yield {
            "nombre": nombre,
            "cargo": cargo,
            "sigla": sigla,
            "depende_de_cargo": (parent or {}).get("cargo"),
            "depende_de_sigla": (parent or {}).get("sigla"),
            "ruta_cargos": " > ".join(current_path),
        }

    current = {"cargo": cargo, "sigla": sigla}
    for child in node.get("dependencias", []):
        yield from iter_people_with_dependency(child, parent=current, path=current_path)


organigram_people = pd.DataFrame(iter_people_with_dependency(organigram))


def search_organigram_people_fuzzy(
    query: str,
    field: str = "nombre",
    limit: int = 5,
    score_cutoff: int = 75,
    cargo_cleaner: str = "role",
) -> pd.DataFrame:
    """Fuzzy-match `query` against `organigram_people[field]`, ranked by similarity.

    Always returns the top `limit` results regardless of `score_cutoff` — the
    cutoff no longer filters rows, it only marks them via `supera_cutoff`, so
    a query that has no good match still shows its best available candidates.

    For `field="cargo"`, `cargo_cleaner` picks how the text is cleaned before
    matching: "role" (default) uses `clean_cargo` (role-prefix extraction +
    GCBA boilerplate removal); "boilerplate" uses `strip_cargo_boilerplate`
    (GCBA boilerplate removal only, keeping the parent-institution chain) as
    an alternate comparison.

    The result includes `texto_consulta`/`texto_comparado`, the cleaned
    strings rapidfuzz actually scored, so a match_score is easy to sanity-check.

    Args:
        query (str): Raw text to search for (nombre or cargo).
        field (str): "nombre" or "cargo".
        limit (int): Max number of results to return.
        score_cutoff (int): Threshold used only to flag `supera_cutoff`; doesn't filter rows.
        cargo_cleaner (str): "role" or "boilerplate" -- which cleaning to
            apply when field="cargo".

    Returns:
        pd.DataFrame: Top `limit` rows with `texto_consulta`, `texto_comparado`,
        `match_score`, and `supera_cutoff`, sorted by score descending.
    """
    if field == "cargo":
        clean = clean_cargo if cargo_cleaner == "role" else strip_cargo_boilerplate
    else:
        clean = normalize_text

    query_normalized = clean(query)
    choices = organigram_people[field].map(clean)

    matches = process.extract(
        query_normalized,
        choices,
        scorer=fuzz.token_set_ratio,
        limit=limit,
        score_cutoff=0,
    )

    matched_indices = [index for _, _, index in matches]
    scores = {index: score for _, score, index in matches}

    result = organigram_people.loc[matched_indices].copy()
    result["texto_consulta"] = query_normalized
    result["texto_comparado"] = choices.loc[matched_indices].values
    result["match_score"] = result.index.map(scores)
    result["supera_cutoff"] = result["match_score"] >= score_cutoff
    return result.sort_values("match_score", ascending=False).reset_index(drop=True)

### Comparación: búsqueda por embeddings (sentence-transformers + BM25)

Reutiliza `aymurai.models.sentence_encoder.factory.create_encoder` y
`aymurai.transforms.entity_subcategories.bm25.BM25Scorer` (sin modificarlos)
para armar un scorer híbrido — el mismo patrón que
`SentenceTransformerSubcategorizer` en
`aymurai/transforms/entity_subcategories/sentence_transformer.py`, pero
recuperando filas de `organigram_people` en vez de subcategorías de la
taxonomía.

Empíricamente, el híbrido BM25+embeddings solo mejora sobre embeddings puros
cuando ambos scorers reciben texto ya limpiado con `clean_cargo`/
`normalize_text` — con texto crudo, BM25 termina premiando el solapamiento
léxico con el boilerplate GCBA y empeora el resultado. Por eso
`search_organigram_people_embeddings` limpia el texto igual que
`search_organigram_people_fuzzy` antes de comparar.

In [ ]:
import numpy as np

from aymurai.models.sentence_encoder.factory import create_encoder
from aymurai.transforms.entity_subcategories.bm25 import BM25Scorer

EMBEDDING_ENCODER = create_encoder(encoder_type="minilm")

_embedding_cache: dict[str, np.ndarray] = {}
_bm25_cache: dict[str, BM25Scorer] = {}


def _l2_normalize(vectors: np.ndarray) -> np.ndarray:
    """L2-normalize each row of a matrix, so the dot product becomes cosine similarity.

    Args:
        vectors (np.ndarray): Array of shape (n, dim) to normalize row-wise.

    Returns:
        np.ndarray: Same shape as `vectors`, each row unit-norm.
    """
    norms = np.linalg.norm(vectors, axis=-1, keepdims=True)
    return vectors / np.clip(norms, 1e-9, None)


def _cleaned_field_texts(field: str) -> list[str]:
    """Clean every organigram_people[field] value, in row order.

    Args:
        field (str): "nombre" or "cargo".

    Returns:
        list[str]: Cleaned text for each organigram row, same order as the DataFrame.
    """
    clean = clean_cargo if field == "cargo" else normalize_text
    return organigram_people[field].map(clean).tolist()


def _get_field_embeddings(field: str) -> np.ndarray:
    """Return (building and caching to disk/memory if needed) the field's embeddings.

    Args:
        field (str): "nombre" or "cargo".

    Returns:
        np.ndarray: Shape (n_rows, dim), one normalized vector per row.
    """
    if field not in _embedding_cache:
        cache_path = Path(f"organigram_{field}_embeddings.npz")
        texts = _cleaned_field_texts(field)
        vectors = None

        if cache_path.exists():
            data = np.load(cache_path, allow_pickle=True)
            if list(data["texts"]) == texts:
                vectors = data["vectors"]

        if vectors is None:
            vectors = EMBEDDING_ENCODER.batch_encode(
                texts, encoder_type="response_encoder", batch_size=256
            )
            np.savez(cache_path, vectors=vectors, texts=np.array(texts, dtype=object))

        _embedding_cache[field] = _l2_normalize(vectors)

    return _embedding_cache[field]


def _get_field_bm25(field: str) -> BM25Scorer:
    """Return (building and caching in memory if needed) the field's BM25Scorer.

    Args:
        field (str): "nombre" or "cargo".

    Returns:
        BM25Scorer: Scorer built over the field's already-cleaned text.
    """
    if field not in _bm25_cache:
        # Texts are already cleaned, so BM25Scorer's normalize_fn is a no-op.
        _bm25_cache[field] = BM25Scorer(_cleaned_field_texts(field), normalize_fn=lambda t: t)
    return _bm25_cache[field]


def _combine_hybrid_scores(
    sim_scores: np.ndarray, bm25_scores: np.ndarray, bm25_weight: float
) -> np.ndarray:
    """Blend cosine similarity and BM25 scores into a single score vector.

    Args:
        sim_scores (np.ndarray): Cosine similarity per row.
        bm25_scores (np.ndarray): BM25 score per row, same order as sim_scores.
        bm25_weight (float): Weight given to bm25_scores after normalizing
            both to [0, 1]; sim_scores gets 1 - bm25_weight. If <= 0, returns
            sim_scores unchanged.

    Returns:
        np.ndarray: Combined score per row.
    """
    if bm25_weight <= 0:
        return sim_scores

    bm25_max = bm25_scores.max() if bm25_scores.size else 0.0
    sim_max = sim_scores.max() if sim_scores.size else 0.0
    bm25_norm = bm25_scores / (bm25_max + 1e-9) if bm25_max > 0 else np.zeros_like(bm25_scores)
    sim_norm = sim_scores / (sim_max + 1e-9) if sim_max > 0 else np.zeros_like(sim_scores)

    return bm25_weight * bm25_norm + (1 - bm25_weight) * sim_norm


def search_organigram_people_embeddings(
    query: str,
    field: str = "cargo",
    limit: int = 10,
    bm25_weight: float = 0.5,
) -> pd.DataFrame:
    """Hybrid sentence-embedding + BM25 search over organigram_people[field].

    Same retrieval pattern as SentenceTransformerSubcategorizer (cosine
    similarity against cached embeddings, optionally blended with BM25 via
    `bm25_weight`), applied to organigram rows instead of taxonomy
    subcategories. For `field="cargo"`, text is cleaned with `clean_cargo`
    (role-prefix extraction + GCBA boilerplate removal), same as
    search_organigram_people_fuzzy's default -- keeping the parent-institution
    chain made this scorer drift toward the parent institution instead of the
    specific office (confirmed on real corpus data).

    Args:
        query (str): Raw text to search for (nombre or cargo).
        field (str): "nombre" or "cargo".
        limit (int): Max number of results to return.
        bm25_weight (float): Weight of the BM25 score in the blend (0 = embeddings only).

    Returns:
        pd.DataFrame: Top `limit` rows with `texto_consulta`, `similarity_score`,
        `bm25_score`, and `match_score`, sorted by match_score descending.
    """
    clean = clean_cargo if field == "cargo" else normalize_text
    query_clean = clean(query)

    field_embeddings = _get_field_embeddings(field)
    query_vector = EMBEDDING_ENCODER.encode([query_clean], encoder_type="question_encoder")[0]
    query_vector = query_vector / max(np.linalg.norm(query_vector), 1e-9)
    sim_scores = field_embeddings @ query_vector

    bm25_scores = (
        _get_field_bm25(field).score_vector(query_clean)
        if bm25_weight > 0
        else np.zeros_like(sim_scores)
    )
    combined = _combine_hybrid_scores(sim_scores, bm25_scores, bm25_weight)

    top_indices = np.argsort(-combined)[:limit]
    result = organigram_people.iloc[top_indices].copy()
    result["texto_consulta"] = query_clean
    result["similarity_score"] = sim_scores[top_indices]
    result["bm25_score"] = bm25_scores[top_indices]
    result["match_score"] = combined[top_indices]
    return result.reset_index(drop=True)


# # Same troublesome cargos from the rapidfuzz comparisons above.
# display(search_organigram_people_embeddings(
#     "Secretario de Deportes del Ministerio de Desarrollo Económico y Producción de la Ciudad Autónoma de Buenos Aires",
#     field="cargo", limit=5,
# ))
# display(search_organigram_people_embeddings(
#     "Directora General Obras en Vías Peatonales, dependiente del Ministerio de Espacio Público e Higiene Urbana del Gobierno de la Ciudad Autónoma de Buenos Aires",
#     field="cargo", limit=5,
# ))

### Prueba sobre un documento

### Sugerencias de destinatario (organigrama + sector)

Para cada `destinatario` extraído (usando `nombre` y `cargo` por separado), se buscan coincidencias difusas contra:
- el organigrama del GCBA (`organigram_GCBA.json`, igual que en `02-Recepients-sector.ipynb`)
- el listado `destinatario_por_sector.csv`

y se muestran combinadas, ordenadas por `match_score` descendente.

In [ ]:
test_result = evaluate_chat_response(
    system_prompt=system_prompt,
    user_prompt=f"Documento: {doc_path.name}" + "\n\n" + document_text,
    model=MODEL,
    validation_model=DataExtraction,
    max_retries=2,
)

print(f"Validation succeeded: {test_result['validation_succeeded']}")
print(f"Attempts: {test_result['num_attempts']}")
if not test_result["validation_succeeded"]:
    print(f"Validation error: {test_result['validation_error']}")
print(f"Input tokens: {test_result['input_tokens']}")
print(f"Output tokens: {test_result['output_tokens']}")
print(f"Speed (tok/s): {test_result['tokens_per_second']:.1f}")

if test_result["validated_response"]:
    extraction = DataExtraction.model_validate(test_result["validated_response"])
    print("\n--- Extracción ---")
    print(extraction.model_dump_json(indent=2))

print(f"Validation succeeded: {test_result['validation_succeeded']}")

TOP_N = 10
SCORE_CUTOFF = 75

if test_result["validation_succeeded"]:
    extraction = DataExtraction.model_validate(test_result["validated_response"])

    for idx, destinatario in enumerate(extraction.destinatarios):
        print(
            f"\n--- Destinatario {idx} "
            f"(destinatario_principal={destinatario.destinatario_principal}) ---\n"
            f"nombre: {destinatario.nombre}\n"
            f"cargo: {destinatario.cargo}\n"
            f"sector: {destinatario.sector}"
        )

        if normalize_text(destinatario.sector) != "gcba":
            print(f"\nSector '{destinatario.sector}' no es GCBA, se omite el cruce con el organigrama.")
            continue

        print(f"\nCoincidencias por nombre (top {TOP_N}):")
        nombre_matches = search_organigram_people_fuzzy(
            destinatario.nombre, field="nombre", limit=TOP_N, score_cutoff=SCORE_CUTOFF
        )
        display(nombre_matches)

        print(f"\nCoincidencias por cargo (top {TOP_N}):")
        cargo_matches = search_organigram_people_fuzzy(
            destinatario.cargo, field="cargo", limit=TOP_N, score_cutoff=SCORE_CUTOFF
        )
        display(cargo_matches)

        print(f"\nCoincidencias por cargo, limpieza solo boilerplate GCBA (top {TOP_N}):")
        cargo_matches_boilerplate = search_organigram_people_fuzzy(
            destinatario.cargo,
            field="cargo",
            limit=TOP_N,
            score_cutoff=SCORE_CUTOFF,
            cargo_cleaner="boilerplate",
        )
        display(cargo_matches_boilerplate)

        print(f"\nCoincidencias por cargo, embeddings + BM25 (top {TOP_N}):")
        cargo_matches_embeddings = search_organigram_people_embeddings(
            destinatario.cargo, field="cargo", limit=TOP_N
        )
        display(cargo_matches_embeddings)
else:
    print(f"Validation error: {test_result['validation_error']}")

### Extracción corregida por consenso de búsquedas

Cruza las 4 búsquedas anteriores (fuzzy por nombre, fuzzy por cargo completo,
fuzzy por cargo solo boilerplate, embeddings+BM25 por cargo) y solo corrige
`nombre`/`cargo` de un destinatario cuando al menos `MIN_VOTES` de esas 4
coinciden en la misma fila del organigrama (mismo `nombre`+`cargo`+`sigla`).
Si no hay consenso (o el sector no es GCBA, o falta `nombre`/`cargo`), se
conserva el destinatario tal como lo extrajo el LLM. La salida final es la
misma `DataExtraction` original con `destinatarios` reemplazado por la
versión corregida.

In [ ]:
MIN_VOTES = 3  # number of independent searches that must agree to accept a correction


def resolve_destinatario(destinatario, top_n=1, score_cutoff=75, min_votes=MIN_VOTES):
    """Cross-checks nombre/cargo against the organigram via 4 independent
    searches (fuzzy nombre, fuzzy cargo role-cleaned, fuzzy cargo
    boilerplate-only, embeddings+BM25 cargo); only corrects nombre/cargo when
    at least `min_votes` of the 4 agree on the same top-1 organigram row.

    Args:
        destinatario: Destinatario as extracted by the LLM (with nombre/cargo/sector).
        top_n (int): How many results to request from each internal search
            (only the first of each is used for voting).
        score_cutoff (int): Threshold passed to the fuzzy searches (doesn't filter results).
        min_votes (int): Minimum of the 4 searches that must agree to accept the correction.

    Returns:
        tuple[Destinatario, dict]: The destinatario (corrected or unchanged)
        and a detail dict {"validado", "fuentes_coincidentes",
        "picks"/"motivo"} describing the decision.
    """
    if normalize_text(destinatario.sector) != "gcba":
        return destinatario.model_copy(), {
            "validado": False, "motivo": "sector no es GCBA", "fuentes_coincidentes": [], "picks": {},
        }

    if not destinatario.nombre or not destinatario.cargo:
        return destinatario.model_copy(), {
            "validado": False, "motivo": "nombre o cargo faltante", "fuentes_coincidentes": [], "picks": {},
        }

    searches = {
        "fuzzy_nombre": search_organigram_people_fuzzy(
            destinatario.nombre, field="nombre", limit=top_n, score_cutoff=score_cutoff
        ),
        "fuzzy_cargo_completo": search_organigram_people_fuzzy(
            destinatario.cargo, field="cargo", limit=top_n, score_cutoff=score_cutoff, cargo_cleaner="role"
        ),
        "fuzzy_cargo_boilerplate": search_organigram_people_fuzzy(
            destinatario.cargo, field="cargo", limit=top_n, score_cutoff=score_cutoff, cargo_cleaner="boilerplate"
        ),
        "embeddings_cargo": search_organigram_people_embeddings(
            destinatario.cargo, field="cargo", limit=top_n
        ),
    }

    top_picks = {source: df.iloc[0] for source, df in searches.items() if len(df)}

    def identity(row):
        return (row["nombre"], row["cargo"], row["sigla"])

    votes = {}
    for source, row in top_picks.items():
        votes.setdefault(identity(row), []).append(source)

    winning_key, sources = max(votes.items(), key=lambda kv: len(kv[1])) if votes else (None, [])
    validado = len(sources) >= min_votes

    if validado:
        winning_row = top_picks[sources[0]]
        corrected = Destinatario(
            nombre=winning_row["nombre"],
            cargo=winning_row["cargo"],
            destinatario_principal=destinatario.destinatario_principal,
            sector=destinatario.sector,
        )
    else:
        corrected = destinatario.model_copy()

    detalle = {
        "validado": validado,
        "fuentes_coincidentes": sources,
        "picks": {source: identity(row) for source, row in top_picks.items()},
    }
    return corrected, detalle


def build_corrected_extraction(extraction, top_n=1, score_cutoff=75, min_votes=MIN_VOTES):
    """Apply resolve_destinatario to every destinatario in an extraction.

    Args:
        extraction: DataExtraction with the list of destinatarios to review.
        top_n (int): See resolve_destinatario.
        score_cutoff (int): See resolve_destinatario.
        min_votes (int): See resolve_destinatario.

    Returns:
        tuple[DataExtraction, list[dict]]: The extraction with corrected
        destinatarios, and the list of validation details (one per destinatario).
    """
    corrected_destinatarios = []
    detalles = []

    for destinatario in extraction.destinatarios:
        corrected, detalle = resolve_destinatario(
            destinatario, top_n=top_n, score_cutoff=score_cutoff, min_votes=min_votes
        )
        corrected_destinatarios.append(corrected)
        detalles.append(detalle)

    corrected_extraction = extraction.model_copy(update={"destinatarios": corrected_destinatarios})
    return corrected_extraction, detalles


if test_result["validation_succeeded"]:
    corrected_extraction, validacion_detalle = build_corrected_extraction(extraction, min_votes=MIN_VOTES)

    print(f"--- Extracci\u00f3n corregida (MIN_VOTES={MIN_VOTES}) ---")
    print(corrected_extraction.model_dump_json(indent=2))

    print("\n--- Detalle de validaci\u00f3n por destinatario ---")
    for idx, detalle in enumerate(validacion_detalle):
        print(
            f"\nDestinatario {idx}: validado={detalle['validado']} "
            f"fuentes={detalle.get('fuentes_coincidentes')} motivo={detalle.get('motivo')}"
        )
        for source, key in detalle.get("picks", {}).items():
            print(f"  {source}: {key}")


## Evaluación sobre el corpus completo

In [ ]:
errors: list[dict] = []
results: list[dict] = load_json(RESULTS_PATH) if RESULTS_PATH.exists() else []
already_processed = {(r["doc_path"], r["model"], r["device"]) for r in results}
print(f"Resultados ya procesados: {len(results)}")

In [ ]:
for doc in tqdm(documents, desc="Extracting information"):
    doc_name = doc.name

    if (doc_name, MODEL, DEVICE) in already_processed:
        tqdm.write(f"Skip (already done): {doc_name}")
        continue

    session = requests.Session()
    extracted = call_extraction_api(session, doc)
    session.close()

    if extracted["status"] != "success":
        tqdm.write(f"Extraction API error for {doc_name}: {extracted['detail']}")
        errors.append({"doc_path": doc_name, "error": extracted["detail"]})
        continue

    text = "\n".join(extracted["detail"]["document"])
    if not text.strip():
        tqdm.write(f"Skip (empty): {doc_name}")
        continue

    tqdm.write(f"Evaluating: {doc_name}")

    try:
        result = evaluate_chat_response(
            system_prompt=system_prompt,
            user_prompt=f"Documento: {doc_name}" + "\n\n" + text,
            model=MODEL,
            options={"num_ctx": 32_768, "num_predict": 8192},
            validation_model=DataExtraction,
            max_retries=2,
        )
        result["doc_path"] = doc_name
        result["device"] = DEVICE

        if not result["validation_succeeded"]:
            tqdm.write(
                f"Validation failed for {doc_name} after {result['num_attempts']} attempts"
            )
            errors.append(
                {
                    "doc_path": doc_name,
                    "model": MODEL,
                    "device": DEVICE,
                    "error": result["validation_error"],
                }
            )
            continue

        results.append(result)
        already_processed.add((doc_name, MODEL, DEVICE))
        save_json(results, RESULTS_PATH)

    except Exception as exc:
        tqdm.write(f"Error evaluating {doc_name}: {exc}")
        errors.append(
            {"doc_path": doc_name, "model": MODEL, "device": DEVICE, "error": str(exc)}
        )

print(f"\nCompletado: {len(results)} resultados, {len(errors)} errores.")

## Exportar recomendaciones (organigrama + LLM)

Para cada destinatario de cada documento procesado, cruza contra el
organigrama con las 4 búsquedas independientes (fuzzy por nombre, fuzzy
por cargo completo, fuzzy por cargo solo boilerplate, embeddings+BM25 por
cargo) y guarda **solo el primer resultado** de cada una en su propia
columna, junto con la salida cruda del LLM (`llm_raw_output`) y los campos
ya parseados de la extracción. Se guarda en:
- `resources/data/restricted/defensoria/results/recomendaciones.csv`
- `resources/data/restricted/defensoria/results/recomendaciones.json`

In [ ]:
RESULTS_FOLDER = Path("../../../resources/data/restricted/defensoria/results")
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)
RECOMENDACIONES_CSV = RESULTS_FOLDER / "recomendaciones.csv"
RECOMENDACIONES_JSON = RESULTS_FOLDER / "recomendaciones.json"

SEARCH_COLUMNS = (
    "busqueda_fuzzy_nombre",
    "busqueda_fuzzy_cargo_completo",
    "busqueda_fuzzy_cargo_boilerplate",
    "busqueda_embeddings_cargo",
)


def top1_summary(df: pd.DataFrame) -> dict | None:
    """Compact dict for the top-1 row of a search result DataFrame, or None if empty.

    Args:
        df (pd.DataFrame): Search result (sorted by score descending).

    Returns:
        dict | None: {"nombre", "cargo", "sigla", "score"} from the first
        row, or None if `df` is empty.
    """
    if df is None or len(df) == 0:
        return None
    row = df.iloc[0]
    return {
        "nombre": row["nombre"],
        "cargo": row["cargo"],
        "sigla": row["sigla"],
        "score": round(float(row["match_score"]), 2),
    }


def buscar_top1_destinatario(destinatario: Destinatario) -> dict:
    """Runs the 4 independent searches for a destinatario and keeps only the
    top-1 result of each -- raw candidates for review, no consensus/correction.

    Args:
        destinatario (Destinatario): Destinatario as extracted by the LLM.

    Returns:
        dict: One key per search in SEARCH_COLUMNS, each holding the result
        of top1_summary (or None if sector isn't GCBA or nombre/cargo is missing).
    """
    if (
        normalize_text(destinatario.sector) != "gcba"
        or not destinatario.nombre
        or not destinatario.cargo
    ):
        return dict.fromkeys(SEARCH_COLUMNS)

    return {
        "busqueda_fuzzy_nombre": top1_summary(
            search_organigram_people_fuzzy(destinatario.nombre, field="nombre", limit=1)
        ),
        "busqueda_fuzzy_cargo_completo": top1_summary(
            search_organigram_people_fuzzy(
                destinatario.cargo, field="cargo", limit=1, cargo_cleaner="role"
            )
        ),
        "busqueda_fuzzy_cargo_boilerplate": top1_summary(
            search_organigram_people_fuzzy(
                destinatario.cargo, field="cargo", limit=1, cargo_cleaner="boilerplate"
            )
        ),
        "busqueda_embeddings_cargo": top1_summary(
            search_organigram_people_embeddings(destinatario.cargo, field="cargo", limit=1)
        ),
    }


raw_results = load_json(RESULTS_PATH)
print(f"Resultados cargados: {len(raw_results)}")

recomendaciones_rows = []

for result in tqdm(raw_results, desc="Cruzando destinatarios con el organigrama"):
    try:
        extraction_data = result.get("validated_response") or DataExtraction.model_validate(
            parse_json_object(result["chat_response"])
        ).model_dump()
    except Exception as exc:
        tqdm.write(f"Error parseando {result.get('doc_path')}: {exc}")
        continue

    base_row = {
        "doc_path": result.get("doc_path"),
        "model": result.get("model"),
        "device": result.get("device"),
        "llm_raw_output": result.get("chat_response"),
        "numero_recomendacion": extraction_data.get("numero_recomendacion"),
        "fecha_recomendacion": extraction_data.get("fecha_recomendacion"),
        "tema": extraction_data.get("tema"),
        "subtema": extraction_data.get("subtema"),
        "datos_personales": extraction_data.get("datos_personales"),
        "contenido_para_publicar": extraction_data.get("contenido_para_publicar"),
    }

    destinatarios = extraction_data.get("destinatarios") or []
    if not destinatarios:
        recomendaciones_rows.append({
            **base_row,
            "destinatario_idx": None,
            "nombre": None,
            "cargo": None,
            "destinatario_principal": None,
            "sector": None,
            **dict.fromkeys(SEARCH_COLUMNS),
        })
        continue

    for idx, dest_data in enumerate(destinatarios):
        destinatario = Destinatario.model_validate(dest_data)
        recomendaciones_rows.append({
            **base_row,
            "destinatario_idx": idx,
            "nombre": destinatario.nombre,
            "cargo": destinatario.cargo,
            "destinatario_principal": destinatario.destinatario_principal,
            "sector": destinatario.sector,
            **buscar_top1_destinatario(destinatario),
        })

recomendaciones_df = pd.DataFrame(recomendaciones_rows)

csv_df = recomendaciones_df.copy()
for col in SEARCH_COLUMNS:
    csv_df[col] = csv_df[col].map(lambda v: json.dumps(v, ensure_ascii=False) if v is not None else "")

csv_df.to_csv(RECOMENDACIONES_CSV, index=False, encoding="utf-8-sig")
save_json(recomendaciones_rows, RECOMENDACIONES_JSON)

print(f"Guardadas {len(recomendaciones_df)} filas (destinatarios) en:")
print(f"  {RECOMENDACIONES_CSV}")
print(f"  {RECOMENDACIONES_JSON}")
recomendaciones_df.head()


## Análisis de resultados

In [ ]:
raw_results = load_json(RESULTS_PATH)
print(f"Total resultados: {len(raw_results)}")

parsed_rows = []
parse_errors = []

for idx, result in enumerate(raw_results):
    try:
        if result.get("validated_response"):
            extraction = result["validated_response"]
        else:
            extraction = DataExtraction.model_validate(
                parse_json_object(result["chat_response"])
            ).model_dump()

        parsed_rows.append(
            {
                "result_idx": idx,
                "doc_path": result.get("doc_path"),
                "model": result.get("model"),
                "device": result.get("device"),
                "input_tokens": result.get("input_tokens"),
                "output_tokens": result.get("output_tokens"),
                "tokens_per_second": result.get("tokens_per_second"),
                "num_attempts": result.get("num_attempts"),
                **extraction,
            }
        )
    except Exception as exc:
        parse_errors.append({"result_idx": idx, "error": str(exc)})

results_df = pd.DataFrame(parsed_rows)
print(f"Parseados: {len(results_df)} | Errores: {len(parse_errors)}")
results_df.head()

In [ ]:
# Distribución por tema
results_df["tema"].value_counts(dropna=False).to_frame("count")

In [ ]:
# Estadísticas de destinatarios: proporción principal vs notificado por documento
def destinatarios_stats(row) -> dict:
    """Count principal vs. notified destinatarios in a results row.

    Args:
        row: Row of results_df (must have a "destinatarios" column with the list).

    Returns:
        dict: {"n_total", "n_principal", "n_notificacion"} for that row.
    """
    dests = row["destinatarios"]
    if not isinstance(dests, list):
        return {"n_total": 0, "n_principal": 0, "n_notificacion": 0}
    n_principal = sum(1 for d in dests if d.get("destinatario_principal"))
    return {
        "n_total": len(dests),
        "n_principal": n_principal,
        "n_notificacion": len(dests) - n_principal,
    }


dest_stats = results_df.apply(destinatarios_stats, axis=1, result_type="expand")
display(dest_stats.describe())
results_df[["doc_path"]].join(dest_stats).sort_values("n_total", ascending=False).head(
    10
)

In [ ]:
# Métricas de performance
perf_cols = ["input_tokens", "output_tokens", "tokens_per_second", "num_attempts"]
results_df[perf_cols].describe(percentiles=[0.25, 0.5, 0.75, 0.90, 0.95, 0.99])

## Análisis de rendimiento: fuzzy vs embeddings vs híbrido

Corre la búsqueda **exactamente como lo hace el endpoint** (`/llm/data-extraction`):
importa directamente `aymurai.api.endpoints.routers.llm.data_extraction.organigram_matching`
en vez de reimplementar la lógica acá, para que este análisis no se desincronice del
código real. Se evalúan las 3 configuraciones de `search_backend`:

- `fuzzy`
- `embeddings`
- `hybrid` con `hybrid_weight=0.5` (50/50)

**Nota sobre el diseño del endpoint:** `search_candidates` ya no combina nombre y cargo
en una única respuesta -- devuelve dos listas independientes, `candidatos_nombre` y
`candidatos_cargo`, cada una rankeada por su propia búsqueda. Esto es intencional: en el
front, cada destinatario tiene dos desplegables separados (uno por nombre, otro por
cargo), y elegir un candidato de nombre no obliga a elegir el candidato de cargo de la
misma fila del organigrama, porque las personas pueden cambiar de cargo.

**Por qué el foco está en `cargo` y no en `nombre`:** el organigrama es una foto de un
momento dado. Cuando cambia el gobierno (o incluso un reacomodo de gabinete), los
**nombres** de las personas en cada cargo cambian, pero la **estructura de cargos/oficinas**
tiende a ser mucho más estable. Una búsqueda que depende del nombre para acertar deja de
funcionar apenas hay un cambio de gestión; una que depende del cargo sigue siendo útil.
Por eso, además de reproducir el comportamiento real del endpoint (dos búsquedas
independientes), esta sección le presta particular atención a la lista de **cargo** para
medir qué tan bien sobreviviría cada backend a un cambio de gobierno.


In [ ]:
import os
import sys

os.environ["RESOURCES_BASEPATH"] = str(Path("../../../resources").resolve())
for module_name in list(sys.modules):
    if module_name == "aymurai.settings" or module_name.startswith("aymurai.api"):
        del sys.modules[module_name]

from aymurai.api.endpoints.routers.llm.data_extraction import organigram_matching as om


om._load_organigram_people = lambda: organigram_people
om._get_field_embeddings.cache_clear()
om._get_field_bm25.cache_clear()

BACKEND_CONFIGS = {
    "fuzzy": {"backend": "fuzzy"},
    "embeddings": {"backend": "embeddings"},
    "hybrid_50_50": {"backend": "hybrid", "hybrid_weight": 0.5},
}
PERF_TOP_K = 5


### Preparar destinatarios GCBA desde el corpus ya extraído

In [ ]:
raw_results = load_json(RESULTS_PATH)
print(f"Documentos con resultados: {len(raw_results)}")

destinatarios_gcba = []
for result in raw_results:
    validated = result.get("validated_response")
    if not validated:
        continue
    for idx, dest in enumerate(validated.get("destinatarios", [])):
        if normalize_text(dest.get("sector")) != "gcba":
            continue
        if not dest.get("nombre") or not dest.get("cargo"):
            continue
        destinatarios_gcba.append(
            {
                "doc_path": result.get("doc_path"),
                "destinatario_idx": idx,
                "nombre": dest["nombre"],
                "cargo": dest["cargo"],
            }
        )

print(f"Destinatarios GCBA con nombre+cargo completos: {len(destinatarios_gcba)}")


### Ejecutar las 3 configuraciones

Para cada destinatario y cada backend se llama una sola vez a `search_candidates` (tal
cual la usa el endpoint), que ya devuelve las dos listas independientes:

1. **`candidatos_nombre`**: ranking de la búsqueda por nombre sola.
2. **`candidatos_cargo`**: ranking de la búsqueda por cargo sola -- la señal que
   sobrevive un cambio de gobierno.

No hace falta reimplementar una búsqueda "cargo solo" por separado: `candidatos_cargo`
ya es exactamente eso.


In [ ]:
_ = om.search_candidates(
    nombre="warmup", cargo="warmup", sector="GCBA", backend="hybrid", top_k=1, hybrid_weight=0.5
)

performance_rows = []

for dest in tqdm(destinatarios_gcba, desc="Evaluando busquedas"):
    for label, cfg in BACKEND_CONFIGS.items():
        backend = cfg["backend"]
        hybrid_weight = cfg.get("hybrid_weight", 0.5)

        start = time.perf_counter()
        candidates = om.search_candidates(
            nombre=dest["nombre"],
            cargo=dest["cargo"],
            sector="GCBA",
            backend=backend,
            top_k=PERF_TOP_K,
            hybrid_weight=hybrid_weight,
        )
        elapsed_ms = (time.perf_counter() - start) * 1000

        nombre_list = candidates["nombre"]
        cargo_list = candidates["cargo"]
        nombre_top1 = nombre_list[0] if nombre_list else None
        cargo_top1 = cargo_list[0] if cargo_list else None
        cargo_top2_score = cargo_list[1].score if len(cargo_list) > 1 else None
        cargo_margin = (
            (cargo_top1.score - cargo_top2_score)
            if cargo_top1 is not None and cargo_top2_score is not None
            else None
        )

        performance_rows.append(
            {
                "doc_path": dest["doc_path"],
                "destinatario_idx": dest["destinatario_idx"],
                "backend": label,
                "nombre_extraido": dest["nombre"],
                "cargo_extraido": dest["cargo"],
                "nombre_top1_nombre": nombre_top1.nombre if nombre_top1 else None,
                "nombre_top1_sigla": nombre_top1.sigla if nombre_top1 else None,
                "nombre_top1_score": nombre_top1.score if nombre_top1 else None,
                "cargo_top1_cargo": cargo_top1.cargo if cargo_top1 else None,
                "cargo_top1_sigla": cargo_top1.sigla if cargo_top1 else None,
                "cargo_top1_score": cargo_top1.score if cargo_top1 else None,
                "cargo_top1_margin": cargo_margin,
                "search_latency_ms": elapsed_ms,
            }
        )

perf_df = pd.DataFrame(performance_rows)
print(f"Filas: {len(perf_df)} ({len(destinatarios_gcba)} destinatarios x {len(BACKEND_CONFIGS)} backends)")
perf_df.head()


### Latencia por backend

In [ ]:
perf_df.groupby("backend")[["search_latency_ms"]].agg(["mean", "median"])


### Acuerdo entre backends (misma sigla en el top-1)

Ahora que el endpoint devuelve dos listas independientes, se compara el acuerdo entre
backends por separado en cada una: primero la lista de **cargo** (la señal que nos
interesa que sea estable) y después la de **nombre**. Si el acuerdo entre backends es
más bajo en nombre que en cargo, es porque el nombre es la señal más frágil -- algo
esperable dado que dos backends distintos pueden "leer" un nombre propio de formas muy
distintas (fuzzy por caracteres, embeddings por semántica), mientras que el cargo tiene
más estructura textual para apoyarse.


In [ ]:
backends = list(BACKEND_CONFIGS)


def pairwise_agreement(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    """Compute top-1 agreement between every pair of backends on a given column.

    Args:
        df (pd.DataFrame): perf_df, with "doc_path", "destinatario_idx",
            "backend", and `value_col` columns.
        value_col (str): Column to compare across backends (e.g. "cargo_top1_sigla").

    Returns:
        pd.DataFrame: One row per backend pair, with "backend_a",
        "backend_b", and "acuerdo_top1" (fraction of destinatarios where both
        agree on `value_col`).
    """
    pivot = df.pivot_table(
        index=["doc_path", "destinatario_idx"], columns="backend", values=value_col, aggfunc="first"
    )
    rows = []
    for i, b1 in enumerate(backends):
        for b2 in backends[i + 1 :]:
            rows.append(
                {"backend_a": b1, "backend_b": b2, "acuerdo_top1": (pivot[b1] == pivot[b2]).mean()}
            )
    return pd.DataFrame(rows)


print("=== Acuerdo top-1 en la lista de CARGO ===")
display(pairwise_agreement(perf_df, "cargo_top1_sigla"))

print("\n=== Acuerdo top-1 en la lista de NOMBRE ===")
display(pairwise_agreement(perf_df, "nombre_top1_sigla"))


### ¿Nombre y cargo apuntan a la misma persona/cargo?

El endpoint ya no elige una única respuesta final combinando nombre y cargo -- el
front muestra las dos listas y el usuario elige de cada una por separado, justamente
porque una persona puede haber dejado de ocupar el cargo que el organigrama le asigna
(o viceversa). Igual es útil medir, para cada backend, en qué fracción de los
destinatarios el top-1 de la lista de **nombre** y el top-1 de la lista de **cargo**
coinciden en la misma fila del organigrama (misma `sigla`). Un desacuerdo alto no es un
error del endpoint -- puede reflejar rotación real de personas -- pero también puede
señalar que la búsqueda por nombre está encontrando coincidencias poco confiables.


In [ ]:
rows = []
for backend in perf_df["backend"].unique():
    sub = perf_df[perf_df["backend"] == backend]
    coincide = (sub["nombre_top1_sigla"] == sub["cargo_top1_sigla"]).mean()
    rows.append({"backend": backend, "nombre_cargo_top1_coinciden": coincide})

pd.DataFrame(rows).set_index("backend")


### Distribución de scores y margen (lista de cargo)

`fuzzy` usa escala 0-100, `embeddings`/`hybrid` usan similitud coseno (~0-1) -- **no son
comparables en magnitud absoluta entre sí**, así que se muestran por separado. El
`margen` (score del top-1 menos el del top-2) es una medida de qué tan clara es la
decisión dentro de cada backend.


In [ ]:
print("=== Score top-1 (lista de cargo) ===")
display(perf_df.groupby("backend")["cargo_top1_score"].describe())

print("\n=== Margen top1-top2 (lista de cargo) ===")
display(perf_df.groupby("backend")["cargo_top1_margin"].describe())

print("\n=== Tasa de saturación (top-1 en el techo de la escala) ===")
saturation_thresholds = {"fuzzy": 99.9, "embeddings": 0.999, "hybrid_50_50": 0.999}
for backend, threshold in saturation_thresholds.items():
    sub = perf_df[perf_df["backend"] == backend]
    rate = (sub["cargo_top1_score"] >= threshold).mean()
    print(f"  {backend:15}: {rate:.1%} de los casos con score >= techo ({threshold})")


### Ejemplo concreto: por qué las listas deben ser independientes

Caso real del corpus (`Resolucion-1440-22...`): con `embeddings`, el top-1 de la lista
de **nombre** termina siendo una persona/cargo completamente ajenos al destinatario
correcto, con un score de similitud engañosamente alto -- mientras que la lista de
**cargo** sí encuentra el cargo correcto. Si ambas señales se combinaran en una única
respuesta (como hacía el diseño anterior), este tipo de caso podía hacer que el nombre
"le ganara" a un cargo que sí era correcto. Mantener las dos listas independientes evita
exactamente este modo de falla: el usuario ve ambas y elige.


In [ ]:
ejemplo = perf_df[
    (perf_df["doc_path"] == "Resolucion-1440-22.-Control-de-obras-habilitaciones-y-permisos.pdf")
    & (perf_df["backend"] == "embeddings")
]
ejemplo[
    [
        "nombre_extraido",
        "cargo_extraido",
        "nombre_top1_nombre",
        "nombre_top1_sigla",
        "nombre_top1_score",
        "cargo_top1_cargo",
        "cargo_top1_sigla",
        "cargo_top1_score",
    ]
]


### Conclusión

Con los datos de este corpus (cifras a re-generar corriendo esta sección con el nuevo
esquema de listas independientes -- las de la corrida anterior correspondían al diseño
"merged" ya discontinuado):

- **El acuerdo entre backends tiende a ser más alto en la lista de cargo que en la de
  nombre** -- confirma que el nombre es la señal más frágil entre backends, no el cargo.
- **Nombre y cargo no siempre apuntan a la misma fila del organigrama** en una fracción
  no despreciable de los destinatarios -- esperable dado que las personas cambian de
  cargo, y es justamente la razón por la que el endpoint ya no fuerza una respuesta
  única combinada.
- **`fuzzy` es órdenes de magnitud más rápido** (sin inferencia de modelo) y su score
  0-100 tiene más rango dinámico (menos saturación en el techo) que `embeddings`, cuya
  similitud coseno tiende a acumularse cerca de 1.0 y perder poder de discriminación.
- **`hybrid_50_50` no domina claramente a los otros dos** en esta muestra: hereda parte
  de la fragilidad de nombre de `embeddings` y no es tan rápido como `fuzzy` puro.

Dado que el cargo es la señal que más nos interesa que sobreviva un cambio de gobierno,
y que `fuzzy` iguala o supera a los otros backends en la lista de cargo con una fracción
del costo, estos resultados no muestran una ventaja clara de embeddings/hybrid que
justifique su costo -- pero esta es una muestra de un solo corpus y un solo momento del
organigrama; vale la pena repetir este análisis cuando haya más documentos procesados o
un organigrama de otra fecha.